# 🏰 Zork GPT Playground
Run your own AI-enhanced text adventure in Google Colab!

### Instructions
1. Run the cell below.
2. Wait for the installation to finish.
3. Click the **public URL** (e.g., `https://xxxx.gradio.live`) that appears at the bottom.
4. Enjoy!

In [ ]:
# Install dependencies
!pip install gradio openai google-generativeai anthropic groq requests pillow diffusers torch transformers accelerate

# Clone the repository (or just create the files if running from scratch)
# For this notebook, we will write the files directly so it works standalone.

import os

# Write providers.py
with open("providers.py", "w") as f:
    f.write('''
import os
import requests
import json
from abc import ABC, abstractmethod

class LLMProvider(ABC):
    @abstractmethod
    def generate(self, system_prompt, history, new_input, model_name=None, api_key=None):
        pass

    @abstractmethod
    def get_models(self, api_key=None):
        pass

class OllamaProvider(LLMProvider):
    def __init__(self, base_url="http://localhost:11434"):
        self.base_url = base_url

    def get_models(self, api_key=None):
        try:
            # In Colab, Ollama is tricky to run. We will return a placeholder or try to connect to a tunnel.
            # But for simplicity, we assume this is mostly for Cloud APIs in Colab.
            return ["Ollama (Requires Local Tunnel)"]
        except Exception as e:
            return [f"Error: {str(e)}"]

    def generate(self, system_prompt, history, new_input, model_name="llama3", api_key=None):
        return "Ollama is not easily supported in standard Colab without setup. Please use OpenAI, Gemini, or Groq."

class OpenAIProvider(LLMProvider):
    def get_models(self, api_key=None):
        return ["gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo"]

    def generate(self, system_prompt, history, new_input, model_name="gpt-4o", api_key=None):
        if not api_key:
            return "Error: API Key required for OpenAI"
        
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        
        messages = [{"role": "system", "content": system_prompt}]
        for h in history:
            messages.append({"role": "user", "content": h[0]})
            messages.append({"role": "assistant", "content": h[1]})
        messages.append({"role": "user", "content": new_input})

        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=messages
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Error: {str(e)}"

class GeminiProvider(LLMProvider):
    def get_models(self, api_key=None):
        return ["gemini-1.5-pro", "gemini-1.5-flash", "gemini-pro"]

    def generate(self, system_prompt, history, new_input, model_name="gemini-1.5-flash", api_key=None):
        if not api_key:
            return "Error: API Key required for Gemini"
        
        import google.generativeai as genai
        genai.configure(api_key=api_key)
        
        try:
            model = genai.GenerativeModel(model_name, system_instruction=system_prompt)
            gemini_history = []
            for h in history:
                gemini_history.append({"role": "user", "parts": [h[0]]})
                gemini_history.append({"role": "model", "parts": [h[1]]})
            
            chat = model.start_chat(history=gemini_history)
            response = chat.send_message(new_input)
            return response.text
        except Exception as e:
            return f"Error: {str(e)}"

class AnthropicProvider(LLMProvider):
    def get_models(self, api_key=None):
        return ["claude-3-opus-20240229", "claude-3-sonnet-20240229", "claude-3-haiku-20240307"]

    def generate(self, system_prompt, history, new_input, model_name="claude-3-haiku-20240307", api_key=None):
        if not api_key:
            return "Error: API Key required for Anthropic"
        
        import anthropic
        client = anthropic.Anthropic(api_key=api_key)
        
        messages = []
        for h in history:
            messages.append({"role": "user", "content": h[0]})
            messages.append({"role": "assistant", "content": h[1]})
        messages.append({"role": "user", "content": new_input})

        try:
            response = client.messages.create(
                model=model_name,
                max_tokens=1024,
                system=system_prompt,
                messages=messages
            )
            return response.content[0].text
        except Exception as e:
            return f"Error: {str(e)}"

class GroqProvider(LLMProvider):
    def get_models(self, api_key=None):
        return ["llama3-8b-8192", "llama3-70b-8192", "mixtral-8x7b-32768", "gemma-7b-it"]

    def generate(self, system_prompt, history, new_input, model_name="llama3-70b-8192", api_key=None):
        if not api_key:
            return "Error: API Key required for Groq"
        
        from groq import Groq
        client = Groq(api_key=api_key)
        
        messages = [{"role": "system", "content": system_prompt}]
        for h in history:
            messages.append({"role": "user", "content": h[0]})
            messages.append({"role": "assistant", "content": h[1]})
        messages.append({"role": "user", "content": new_input})

        try:
            chat_completion = client.chat.completions.create(
                messages=messages,
                model=model_name,
            )
            return chat_completion.choices[0].message.content
        except Exception as e:
            return f"Error: {str(e)}"
''')

# Write image_gen.py
with open("image_gen.py", "w") as f:
    f.write('''
import os
from abc import ABC, abstractmethod
try:
    from huggingface_hub import hf_hub_download
except ImportError:
    hf_hub_download = None

class ImageProvider(ABC):
    @abstractmethod
    def generate_image(self, prompt, api_key=None):
        pass

class OpenAIImageProvider(ImageProvider):
    def generate_image(self, prompt, api_key=None):
        if not api_key:
            return None
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        try:
            response = client.images.generate(
                model="dall-e-3",
                prompt=f"Retro text adventure game art, fantasy style. {prompt}",
                size="1024x1024",
                quality="standard",
                n=1,
            )
            return response.data[0].url
        except Exception as e:
            print(f"OpenAI Image Error: {e}")
            return None

class GeminiImageProvider(ImageProvider):
    def generate_image(self, prompt, api_key=None):
        return None 

class LocalStableDiffusionProvider(ImageProvider):
    def __init__(self):
        self.pipe = None

    def load_model(self):
        if self.pipe is None:
            try:
                import torch
                from diffusers import StableDiffusionXLPipeline, UNet2DConditionModel, EulerDiscreteScheduler
                from huggingface_hub import hf_hub_download
                base = "stabilityai/stable-diffusion-xl-base-1.0"
                repo = "ByteDance/SDXL-Lightning"
                ckpt = "sdxl_lightning_4step_unet.safetensors"
                
                unet = UNet2DConditionModel.from_config(base, subfolder="unet").to("cuda", torch.float16)
                unet.load_state_dict(torch.load(hf_hub_download(repo, ckpt), map_location="cuda"))
                
                self.pipe = StableDiffusionXLPipeline.from_pretrained(base, unet=unet, torch_dtype=torch.float16, variant="fp16").to("cuda")
                self.pipe.scheduler = EulerDiscreteScheduler.from_config(self.pipe.scheduler.config, timestep_spacing="trailing")
            except Exception as e:
                print(f"Failed to load local SD: {e}")

    def generate_image(self, prompt, api_key=None):
        try:
            if self.pipe is None:
                self.load_model()
            if self.pipe is None:
                 return None
            
            image = self.pipe(prompt, num_inference_steps=4, guidance_scale=0).images[0]
            return image
        except Exception:
            return None
''')

# Write backend.py
with open("backend.py", "w") as f:
    f.write('''
import providers
import image_gen

class GameEngine:
    def __init__(self):
        self.providers = {
            "Ollama": providers.OllamaProvider(),
            "OpenAI": providers.OpenAIProvider(),
            "Gemini": providers.GeminiProvider(),
            "Claude": providers.AnthropicProvider(),
            "Groq": providers.GroqProvider()
        }
        self.image_providers = {
            "OpenAI": image_gen.OpenAIImageProvider(),
            "Gemini": image_gen.GeminiImageProvider(),
            "Local (SD)": image_gen.LocalStableDiffusionProvider()
        }
        
        self.default_system_prompt = """You are the Dungeon Master for a text adventure game based on Zork 1. 
Your goal is to simulate the game world, describe locations, track inventory, and handle user commands.
Be descriptive, atmospheric, and immersive. 
If the user asks to do something impossible, explain why.
Keep the tone consistent with the original Zork: mysterious, slightly witty, and perilous.
Start by describing the opening scene: "West of House".
Do not break character.
"""

    def get_provider(self, name):
        return self.providers.get(name)

    def get_image_provider(self, name):
        return self.image_providers.get(name)

    def chat(self, message, history, text_provider_name, image_provider_name, 
             text_model, text_api_key, image_api_key, system_prompt):
        
        if not system_prompt:
            system_prompt = self.default_system_prompt

        # 1. Generate Text
        provider = self.get_provider(text_provider_name)
        if not provider:
            return "Error: Invalid Text Provider", None
        
        response_text = provider.generate(
            system_prompt, 
            history, 
            message, 
            model_name=text_model, 
            api_key=text_api_key
        )

        # 2. Generate Image (Optional)
        image_url = None
        if image_provider_name and image_provider_name != "None":
            img_prov = self.get_image_provider(image_provider_name)
            if img_prov:
                image_prompt = response_text[:300] 
                image_url = img_prov.generate_image(image_prompt, api_key=image_api_key)

        return response_text, image_url
''')

# Write app.py
with open("app.py", "w") as f:
    f.write('''
import gradio as gr
import backend

engine = backend.GameEngine()

def get_models_for_provider(provider_name, api_key):
    provider = engine.get_provider(provider_name)
    if provider:
        return gr.update(choices=provider.get_models(api_key), value=provider.get_models(api_key)[0] if provider.get_models(api_key) else None)
    return gr.update(choices=[], value=None)

def game_turn(message, history, text_prov, img_prov, txt_model, txt_key, img_key, sys_prompt):
    response, image = engine.chat(
        message, 
        history, 
        text_prov, 
        img_prov, 
        txt_model, 
        txt_key, 
        img_key, 
        sys_prompt
    )
    return response

with gr.Blocks(title="Zork GPT Playground", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏰 Zork GPT Playground")
    gr.Markdown("Play Zork enhanced by LLMs. Choose your provider and model below.")
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### ⚙️ Settings")
            
            # Text Provider Settings
            text_provider = gr.Dropdown(
                choices=["Ollama", "OpenAI", "Gemini", "Claude", "Groq"], 
                value="Ollama", 
                label="Text Provider"
            )
            text_api_key = gr.Textbox(label="Text API Key", type="password", placeholder="Leave empty for Ollama")
            text_model = gr.Dropdown(label="Model", allow_custom_value=True)
            
            refresh_btn = gr.Button("🔄 Refresh Models")
            
            gr.Markdown("---")
            
            # Image Provider Settings
            image_provider = gr.Dropdown(
                choices=["None", "OpenAI", "Gemini", "Local (SD)"], 
                value="None", 
                label="Image Provider"
            )
            image_api_key = gr.Textbox(label="Image API Key", type="password")
            
            gr.Markdown("---")
            system_prompt = gr.Textbox(
                label="System Prompt", 
                value=engine.default_system_prompt,
                lines=5
            )

        with gr.Column(scale=3):
            scene_image = gr.Image(label="Current Scene", interactive=False, height=300)
            chatbot = gr.Chatbot(height=500, type="messages")
            msg = gr.Textbox(label="Your Command", placeholder="open mailbox, go north...")
            clear = gr.Button("Clear")

            history = gr.State([])

            def user(user_message, history):
                return "", history + [{"role": "user", "content": user_message}]

            def bot(history, text_prov, img_prov, txt_model, txt_key, img_key, sys_prompt):
                user_message = history[-1]["content"]
                old_history_format = []
                for i in range(0, len(history)-1, 2):
                    if i+1 < len(history):
                        old_history_format.append([history[i]["content"], history[i+1]["content"]])
                
                response_text, image_url = engine.chat(
                    user_message, 
                    old_history_format, 
                    text_prov, 
                    img_prov, 
                    txt_model, 
                    txt_key, 
                    img_key, 
                    sys_prompt
                )
                
                history.append({"role": "assistant", "content": response_text})
                return history, image_url

            msg.submit(user, [msg, history], [msg, history], queue=False).then(
                bot, 
                [history, text_provider, image_provider, text_model, text_api_key, image_api_key, system_prompt], 
                [chatbot, scene_image]
            )
            clear.click(lambda: None, None, chatbot, queue=False)

    text_provider.change(get_models_for_provider, [text_provider, text_api_key], [text_model])
    refresh_btn.click(get_models_for_provider, [text_provider, text_api_key], [text_model])

if __name__ == "__main__":
    demo.launch(share=True)
''')

!python app.py
